# kbprojection NLI Showcase

Use this notebook to inspect natural language inference examples, edit the prompt, and run the kbprojection pipeline on a small configurable set of problems.

The run starts with a no-KB LangPro baseline, asks an LLM for candidate KB injections when needed, and compares raw versus normalised KB results.

## 1. Load the repo

In Colab this clones the repo from GitHub. In a local Jupyter session it uses the current checkout, so start Jupyter from this repo or from the `notebooks/` directory.

Change `REPO_URL` or `REF` if you want Colab to run a fork, branch, tag, or commit.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/ettoc00/kbprojection.git"
REF = ""  # Optional branch/tag/commit, for example "main" or a commit SHA.

def find_local_repo_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "kbprojection").is_dir():
            return candidate
    raise RuntimeError("Could not find the kbprojection repo. Start Jupyter from the repo or set REPO_DIR manually.")

if IN_COLAB:
    REPO_DIR = Path("/content/kbprojection")
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f"Repo already exists at {REPO_DIR}; fetching updates.")
        !git -C {REPO_DIR} fetch --all --tags
    if REF:
        !git -C {REPO_DIR} checkout {REF}
else:
    REPO_DIR = find_local_repo_dir(Path.cwd().resolve())
    if REF:
        print("REF is ignored in local mode; use git directly in your checkout.")

os.chdir(REPO_DIR)
!{sys.executable} -m pip install -q -e ".[notebook]"

print("Runtime:", "Colab" if IN_COLAB else "local")
print("Repository ready:", REPO_DIR)
!git -C {REPO_DIR} rev-parse --short HEAD

## 2. Configure secrets and demo settings

In Colab, add secrets named `OPENROUTER_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY`, or `ANTHROPIC_API_KEY`. The defaults below use OpenRouter, but you can switch provider/model here.

By default, the notebook uses the repo's default LangPro endpoint. To run locally, set `RUN_LOCAL_LANGPRO_SETUP = True` in the setup cell below.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

for secret_name in ["OPENROUTER_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"]:
    if not os.environ.get(secret_name) and userdata is not None:
        try:
            value = userdata.get(secret_name)
        except Exception:
            value = None
        if value:
            os.environ[secret_name] = value

runtime_root = Path("/content") if IN_COLAB else REPO_DIR / ".notebook-runtime"
os.environ.setdefault("KBPROJECTION_APP_DIR", str(runtime_root / "kbprojection-runtime"))
os.environ.setdefault("KBPROJECTION_RESULTS_DIR", str(runtime_root / "kbprojection-results"))
os.environ.setdefault("KBPROJECTION_CACHE_DIR", str(runtime_root / "kbprojection-cache"))

from kbprojection.settings import get_langpro_settings

MODEL = "openai/gpt-5.4-mini"
PROVIDER = "openrouter"  # Use None to let provider inference choose from available keys.
PROMPT_STYLE = "icl"     # Try "legacy_icl", "icl", or "cot".

MAX_PROBLEMS = 3
SELECTED_PROBLEM_IDS = []  # Optional exact IDs from the loaded problem table. Empty means first MAX_PROBLEMS.

LLM_CONCURRENCY = 1
LANGPRO_CONCURRENCY = 2
LOCAL_LANGPRO_CONCURRENCY = 1
JOB_CONCURRENCY = 2
DISCARD_PROVER_CALLS = True
VERBOSE = True

Path(os.environ["KBPROJECTION_RESULTS_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["KBPROJECTION_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

available_keys = [name for name in ["OPENROUTER_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"] if os.environ.get(name)]
print("Configured model:", MODEL)
print("Configured provider:", PROVIDER)
print("Prompt style:", PROMPT_STYLE)
print("LangPro endpoint:", get_langpro_settings().endpoint)
print("Available provider keys:", available_keys)
print("Results dir:", os.environ["KBPROJECTION_RESULTS_DIR"])

### Optional: install local LangPro dependencies

Run this to use local LangPro. It installs SWI-Prolog, vendors EasyCCG under the kbprojection app-data directory, clones LangPro, and then calls `enable_local(...)` once.

In [ ]:
RUN_LOCAL_LANGPRO_SETUP = False

if RUN_LOCAL_LANGPRO_SETUP:
    if IN_COLAB:
        !apt-get update -qq
        !apt-get install -y -qq swi-prolog
        !{sys.executable} -m pip install -q spacy
        !{sys.executable} -m spacy download en_core_web_sm -q

    from kbprojection.easyccg_vendor import install_local_easyccg
    from kbprojection.local_easyccg import easyccg_is_available
    from kbprojection.settings import enable_local, get_langpro_settings
    from kbprojection.settings import resolve_local_langpro_root

    easyccg_dir = install_local_easyccg()
    enable_local(auto_clone=True, easyccg_dir=easyccg_dir)
    local_root = resolve_local_langpro_root(allow_clone=True)
    settings = get_langpro_settings()
    print("Endpoint:", settings.endpoint)
    print("LangPro root:", local_root, "exists=", local_root.exists())
    print("EasyCCG dir:", settings.local_easyccg_dir, "available=", easyccg_is_available(settings.local_easyccg_dir))
    print("Auto-clone:", settings.local_auto_clone)
else:
    print("Skipped. Set RUN_LOCAL_LANGPRO_SETUP = True only when using local LangPro.")

## 3. Test LangPro

Run this before the full showcase when changing LangPro settings. It makes one small prover call using the currently configured endpoint and prints the exact error if the prover is not ready.

In [ ]:
import asyncio
import shutil

from kbprojection.langpro import langpro_api_call
from kbprojection.local_easyccg import easyccg_is_available
from kbprojection.settings import get_langpro_settings

RUN_LANGPRO_SMOKE_TEST = True
SMOKE_PREMISES = ["A dog is running."]
SMOKE_HYPOTHESIS = "An animal is moving."

async def run_langpro_smoke_test():
    settings = get_langpro_settings()
    print("Endpoint:", settings.endpoint)
    if settings.endpoint.startswith("local://"):
        print("LangPro root:", settings.local_root, "exists=", settings.local_root.exists())
        print("swipl:", shutil.which(settings.local_swipl))
        print("java:", shutil.which("java"))
        print("EasyCCG dir:", settings.local_easyccg_dir, "available=", easyccg_is_available(settings.local_easyccg_dir))
        print("spaCy model:", settings.local_easyccg_spacy_model)

    result = await langpro_api_call(SMOKE_PREMISES, SMOKE_HYPOTHESIS, report=True)
    print("Smoke result label:", result.label)
    if result.error:
        print("Smoke result error:", result.error)
    print("Raw response keys:", sorted(result.model_fields_set))
    return result

if RUN_LANGPRO_SMOKE_TEST:
    smoke_result = await run_langpro_smoke_test()
else:
    print("Skipped LangPro smoke test.")

## 4. Load and inspect the NLI problems

This loads a small curated set of SNLI and SICK examples for interactive inspection.

In [ ]:
import pandas as pd
from kbprojection.loaders.sick import SICKLoader
from kbprojection.loaders.snli import SNLILoader
SHOWCASE_PROBLEM_IDS = {
    "sick": {
        "train": [
            "1792", "2281", "2809", "3853", "4181", "4766", "4974", "5149",
            "5198", "5435", "5462", "5554", "5555", "6584", "8624",
        ],
        "dev": ["3586"],
        "test": ["1262", "1497", "3641", "3974", "4297", "4494", "4589", "4972", "5230"],
    },
    "snli": {
        "train": [
            "3629664676.jpg#4r1e",
            "vg len26r4e",
            "vg len66r1e",
            "3159569570.jpg#4r1e",
            "4294390957.jpg#3r1e",
            "1184967930.jpg#4r4e",
            "2521878609.jpg#4r1e",
            "303607405.jpg#4r3e",
            "436393371.jpg#3r1e",
            "vg len84r1e",
            "326456451.jpg#4r1e",
            "vg len120r5e",
            "2543017787.jpg#4r1e",
            "2647049174.jpg#4r1e",
            "2618866067.jpg#4r1e",
            "vg len47r3e",
            "vg len47r4e",
            "vg len46r5e",
            "3228793611.jpg#4r1e",
            "3134092148.jpg#3r2e",
            "2217728745.jpg#4r1e",
            "3479245321.jpg#4r1e",
            "2741990005.jpg#4r1e",
            "3614595423.jpg#4r1e",
            "207584893.jpg#3r1e",
        ],
    },
}
def load_showcase_problems():
    loaded = []
    sick_loader = SICKLoader()
    for split, problem_ids in SHOWCASE_PROBLEM_IDS["sick"].items():
        sick_loader.load(splits=[split])
        loaded.extend(sick_loader.get_problem(problem_id, split=split) for problem_id in problem_ids)
    snli_loader = SNLILoader()
    for split, problem_ids in SHOWCASE_PROBLEM_IDS["snli"].items():
        snli_loader.load(splits=[split])
        loaded.extend(snli_loader.get_problem(problem_id, split=split) for problem_id in problem_ids)
    return loaded
problems = load_showcase_problems()
problems_by_id = {problem.id: problem for problem in problems}
problem_table = pd.DataFrame(
    [
        {
            "id": problem.id,
            "dataset": problem.dataset,
            "split": problem.split,
            "gold_label": problem.gold_label.value,
            "premise": "\n".join(problem.premises),
            "hypothesis": problem.hypothesis,
        }
        for problem in problems
    ]
)
print(f"Loaded {len(problems)} showcase problems.")
display(problem_table)


## 5. Prompt editor

Edit the selected prompt, then click **Save prompt override**. The run cell below uses the edited prompt in this notebook kernel.

In [ ]:
from IPython.display import display
from ipywidgets import Button, Dropdown, HBox, Output, Textarea, VBox

import kbprojection.prompts as prompt_registry

EDITABLE_PROMPTS = prompt_registry.list_prompts()

def get_prompt_entry(name):
    for entry in prompt_registry.prompts["prompts"]:
        if entry["name"] == name:
            return entry
    raise KeyError(name)

initial_prompt = PROMPT_STYLE if PROMPT_STYLE in EDITABLE_PROMPTS else EDITABLE_PROMPTS[0]
prompt_picker = Dropdown(options=EDITABLE_PROMPTS, value=initial_prompt, description="Prompt")
prompt_text = Textarea(
    value=get_prompt_entry(initial_prompt)["template"],
    layout={"width": "100%", "height": "420px"},
)
save_button = Button(description="Save prompt override", button_style="primary")
use_button = Button(description="Use for run")
reset_button = Button(description="Reload selected prompt")
editor_output = Output()

def load_selected_prompt(*_):
    prompt_text.value = get_prompt_entry(prompt_picker.value)["template"]

def save_selected_prompt(*_):
    get_prompt_entry(prompt_picker.value)["template"] = prompt_text.value
    with editor_output:
        editor_output.clear_output()
        print(f"Saved override for {prompt_picker.value}.")

def use_selected_prompt(*_):
    global PROMPT_STYLE
    PROMPT_STYLE = prompt_picker.value
    with editor_output:
        editor_output.clear_output()
        print(f"Run prompt style set to {PROMPT_STYLE}.")

prompt_picker.observe(load_selected_prompt, names="value")
reset_button.on_click(load_selected_prompt)
save_button.on_click(save_selected_prompt)
use_button.on_click(use_selected_prompt)

display(VBox([HBox([prompt_picker, save_button, use_button, reset_button]), prompt_text, editor_output]))

## 6. Select problems for the showcase run

Use exact IDs in `SELECTED_PROBLEM_IDS`, or leave it empty to run the first `MAX_PROBLEMS` examples from the table.

In [ ]:
if SELECTED_PROBLEM_IDS:
    selected_problems = [problems_by_id[problem_id] for problem_id in SELECTED_PROBLEM_IDS]
else:
    selected_problems = problems[:MAX_PROBLEMS]

print(f"Selected {len(selected_problems)} problems:")
settings = get_langpro_settings()
if settings.endpoint.startswith("local://") and any(problem.dataset == "snli" for problem in selected_problems):
    print("Warning: local raw-text parsing is required for SNLI unless a generated local SNLI corpus is available. Run the LangPro smoke test first.")
display(problem_table[problem_table["id"].isin([problem.id for problem in selected_problems])])

## 7. Execute the showcase run

This runs the normal kbprojection pipeline for the selected problems with one prompt style. It tests the no-KB baseline, generates KB injections if needed, and evaluates raw and normalised KBs.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from kbprojection import arun_problems
from kbprojection.runners import infer_provider
from kbprojection.utils import sanitize_filename_part
provider = infer_provider(MODEL, PROVIDER)
showcase_results = await arun_problems(
    selected_problems,
    model=MODEL,
    provider=provider,
    prompt_style=PROMPT_STYLE,
    test_mode="both",
    concurrency=JOB_CONCURRENCY,
    llm_concurrency=LLM_CONCURRENCY,
    langpro_concurrency=LANGPRO_CONCURRENCY,
    local_langpro_concurrency=LOCAL_LANGPRO_CONCURRENCY,
    discard_prover_calls=DISCARD_PROVER_CALLS,
    verbose=VERBOSE,
)
safe_model = sanitize_filename_part(MODEL)
safe_prompt = sanitize_filename_part(PROMPT_STYLE)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_path = (
    Path(os.environ["KBPROJECTION_RESULTS_DIR"])
    / f"nli_showcase__{provider}__{safe_model}__{safe_prompt}__{timestamp}.json"
)
output_payload = {
    "meta": {
        "created_at": timestamp,
        "model": MODEL,
        "provider": provider,
        "prompt_style": PROMPT_STYLE,
        "problem_ids": [problem.id for problem in selected_problems],
        "source_problem_pool": "curated_nli_showcase",
    },
    "results": showcase_results,
}
output_path.write_text(json.dumps(output_payload, indent=2), encoding="utf-8")
print("Wrote:", output_path)
for item in showcase_results:
    problem = item["problem"]
    print("\n", problem["id"], "gold=", problem["gold_label"], "final=", item["final_status"])
    print("  no_kb:", item.get("pred_no_kb"))
    print("  raw:", item.get("pred_with_raw_kb"), item.get("kb_raw"))
    print("  normalised:", item.get("pred_with_kb"), item.get("kb_filtered"))


## 8. Inspect results as a table

In [ ]:
rows = []
for item in showcase_results:
    problem = item["problem"]
    rows.append(
        {
            "id": problem["id"],
            "gold": problem["gold_label"],
            "no_kb": item.get("pred_no_kb"),
            "raw_kb": item.get("pred_with_raw_kb"),
            "normalised_kb": item.get("pred_with_kb"),
            "final_status": item.get("final_status"),
            "kb_raw": "\n".join(item.get("kb_raw") or []),
            "kb_filtered": "\n".join(item.get("kb_filtered") or []),
        }
    )

display(pd.DataFrame(rows))